# Negflo with missing data

Subtracting routed upstream flow from downstream gauged flow gives a **residual** inflow, which can go negative
because the routing never matches the real system exactly. Clipping the negatives to zero inflates the residual
volume, so `Negflo` offers several ways of **redistributing** the negatives by reducing positive flows instead.

`Negflo` is bulum's implementation of NEGFLO6, a Queensland hydrology program for this task. The method names (RW1,
CL1 and SM1 to SM7) are NEGFLO6's output file extensions.

This notebook runs every method on one short example containing missing data, and shows all the results side by side
in a single table against the raw residual. Flows are in ML/d and totals in ML.

A few terms used throughout:

* **Flow limit**: a threshold (ML/d, default 0) given to `Negflo`. Only flow above it counts as part of a flow event,
  and flows are never reduced below it.
* **Flow event**: a run of consecutive days with flow above the flow limit.
* **Absorbing**: an event takes on negative volume by scaling down each flow's excess above the flow limit, in
  proportion. If the event is too small, its flows are set to the flow limit and the rest of the negative volume is
  left over.
* **Unallocated negatives**: negative volume that no event absorbed, reported per column in `unallocated_negatives`
  (for the most recent method run). It is a negative number, and the result's total exceeds the residual's total by
  that amount: e.g. 14 ML unallocated is reported as -14, and a residual of 220 ML gives a result of 234 ML.

**Missing data** (NaN) contributes nothing to any total, ends a flow event, stays missing in the result, and is
otherwise treated like a day of zero flow: negatives on either side of a gap are added together.

In [1]:
import logging

import numpy as np
import pandas as pd

import bulum
from bulum.stats import Negflo

logging.getLogger("bulum").setLevel(logging.WARNING)  # hide info messages about unallocated negatives
print('bulum version: ', bulum.__version__)

bulum version:  0.3.4


## Example data

Measured (downstream) and modelled (routed upstream) flows over 45 days. The measured record has three gaps. In one of
them the gauge failed partway through an event, which splits that event in two.
`Negflo.from_flows` computes the residual as measured - modelled, and marks a day missing if either input is missing.
A single `Negflo` object can run every method; the SM6 segments are given here too, and only SM6 uses them.

In [2]:
nan = np.nan
dates = pd.date_range("2000-01-01", periods=45)
residual = np.array([
    -4, -6, -3, -1,                 # negatives before the first event
    5, 18, 30, 22, 12, 6,           # event A
    -5, -8, -4,                     # negatives ...
    nan, nan, nan, nan,             # ... a gap ...
    -3, -2, 0,                      # ... and more negatives
    10, 35, 60,                     # event B, interrupted by a gap
    nan, nan,
    35, 20, 9,                      # event C (the rest of the interrupted event)
    -6, -10, -7, -2,
    4, 7, 3, 1,                     # event D (small)
    -1, -2, 0,
    nan, nan, nan,
    6, 3,                           # event E (small)
    -2,                             # negatives after the last event
])
modelled = pd.DataFrame({"flow": np.round(20 + 8 * np.sin(np.arange(45) * 2 * np.pi / 30), 1)}, index=dates)
measured = modelled + residual[:, None]   # NaN on the missing days

segments = [(dates[0], dates[19]), (dates[20], dates[-1])]   # SM6: 1-20 Jan and 21 Jan onwards
negflo = Negflo.from_flows(measured, modelled, segments=segments)
raw = negflo.rw1()["flow"]
print(f"Total positive residual: {raw[raw > 0].sum():.0f} ML")
print(f"Total negative residual: {raw[raw < 0].sum():.0f} ML")
print(f"Total residual:          {raw.sum():.0f} ML")
print(f"Missing days:            {raw.isna().sum()}")

Total positive residual: 286 ML
Total negative residual: -66 ML
Total residual:          220 ML
Missing days:            9


## All methods against the raw residual

Each column is the result of one method, alongside the raw residual (RW1), in ML/d. The **Event** column labels the
flow events A to E (with the default flow limit of zero) referred to in the explanations below. Missing days are
greyed out.

The last two columns are variations on SM2, explained at the end: one with a flow limit of 5 ML/d, and one that treats
missing data as zero flow, as NEGFLO6 did.

The bottom rows give, for each column, the total of the result, the total of the residual it started from, and the
unallocated negatives (in ML). The last row is a check: the residual total minus the (negative) unallocated
negatives equals the total of the result.

In [3]:
runs = [
    ("RW1 (raw)", negflo, "rw1"),
    ("CL1", negflo, "cl1"),
    ("SM1", negflo, "sm1"),
    ("SM2", negflo, "sm2"),
    ("SM3", negflo, "sm3"),
    ("SM4", negflo, "sm4"),
    ("SM5", negflo, "sm5"),
    ("SM6", negflo, "sm6"),
    ("SM7", negflo, "sm7"),
    ("SM2, flow limit 5", Negflo.from_flows(measured, modelled, flow_limit=5), "sm2"),
    ("SM2, gaps as zero", Negflo.from_flows(measured.fillna(0), modelled), "sm2"),
]
results, unallocated, residual_totals = {}, {}, {}
for name, nf, method in runs:
    results[name] = getattr(nf, method)()["flow"]
    unallocated[name] = nf.unallocated_negatives["flow"]   # refers to the method just run
    residual_totals[name] = nf.rw1()["flow"].sum()
table = pd.DataFrame(results)

# label the flow events (runs of positive residual)
is_event = (raw > 0).astype(int)
event_number = (is_event.diff().fillna(is_event) == 1).cumsum()
table.insert(0, "Event", np.where(is_event == 1, event_number.map(lambda n: "ABCDEFGH"[n - 1]), ""))
table.index = table.index.strftime("%d %b")

summary = pd.DataFrame([table.drop(columns="Event").sum(), pd.Series(residual_totals), pd.Series(unallocated)],
                       index=["Total", "Residual total", "Unallocated negatives"])
summary.loc["Check: residual total - unallocated"] = summary.loc["Residual total"] - summary.loc["Unallocated negatives"]
summary.insert(0, "Event", "")
full = pd.concat([table, summary])

missing_rows = list(table.index[raw.isna().to_numpy()])
summary_rows = list(summary.index)
(full.style
     .format("{:.1f}", subset=full.columns[1:], na_rep="missing")
     .set_properties(subset=(missing_rows, full.columns), **{"background-color": "#ebeae6", "color": "#8a8984"})
     .set_properties(subset=(summary_rows, full.columns), **{"font-weight": "bold", "border-top": "1px solid #c8c7c2"})
     .set_properties(subset=(full.index, ["RW1 (raw)"]), **{"border-right": "2px solid #c8c7c2"})
     .set_table_styles([{"selector": "th", "props": "text-align: right; padding: 2px 8px;"},
                        {"selector": "th.col_heading", "props": "max-width: 6em; vertical-align: bottom;"},
                        {"selector": "td", "props": "text-align: right; padding: 2px 8px;"}]))

,Event,RW1 (raw),CL1,SM1,SM2,SM3,SM4,SM5,SM6,SM7,"SM2, flow limit 5","SM2, gaps as zero"
01 Jan,,-4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
02 Jan,,-6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
03 Jan,,-3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
04 Jan,,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
05 Jan,A,5.0,5.0,3.8,4.2,4.2,3.8,3.8,3.1,4.2,5.0,4.2
06 Jan,A,18.0,18.0,13.8,15.3,15.3,13.7,13.7,11.0,15.3,15.1,15.3
07 Jan,A,30.0,30.0,23.1,25.5,25.5,22.9,22.9,18.4,25.5,24.4,25.5
08 Jan,A,22.0,22.0,16.9,18.7,18.7,16.8,16.8,13.5,18.7,18.2,18.7
09 Jan,A,12.0,12.0,9.2,10.2,10.2,9.2,9.2,7.4,10.2,10.4,10.2
10 Jan,A,6.0,6.0,4.6,5.1,5.1,4.6,4.6,3.7,5.1,5.8,5.1


## What each method did

The event sizes in the raw residual are A 93 ML, B 105 ML, C 64 ML, D 15 ML and E 9 ML, and there are 66 ML of
negatives in total.

**CL1** simply sets the negatives to zero. That's easy, but the total is now the **positive** residual, so it overstates
the residual volume by the full 66 ML.

**SM1** scales all positive flows by the same factor, `1 - total negative / total positive`, so every event shrinks by
the same proportion and the total matches the residual (no unallocated negatives). The missing days take no part in
either total. SM1 ignores the flow limit.

**SM2** (into the following event, with carry-over): each event absorbs the negatives **before** it.

* Event A absorbs the 14 ML of negatives at the start.
* The 22 ML of negatives after event A, on both sides of the first gap, are added together (the gap counts as a day of
  zero flow) and absorbed by event B. This happens with or without carry-over.
* The gap during the large event splits it into events B and C. Nothing negative lies between them, so event C is
  untouched.
* The 25 ML before event D is more than D's 15 ML, so D is reduced to zero and the remaining 10 ML carries over. With
  the 3 ML just before event E, that's 13 ML against E's 9 ML, so E is also reduced to zero with 4 ML left over.
* Those 4 ML, plus the 2 ML at the end with no following event, are left unallocated (6 ML).

**SM3** is SM2 without carry-over. Event D is still reduced to zero, but its 10 ML shortfall is discarded, so event E
only absorbs the 3 ML just before it and survives. The discarded volume shows up as larger unallocated negatives
(10 + 2 = 12 ML).

**SM4** (into the preceding event, with carry-over): each event absorbs the negatives **after** it.

* The 14 ML at the start has no preceding event, so (as in NEGFLO6) it is not redistributed: it is left unallocated,
  and the total is 14 ML more than the residual.
* Event A absorbs the 22 ML after it (again on both sides of the gap).
* Event B is followed only by a gap, so it's untouched.
* Event C absorbs the 25 ML after it, D the 3 ML after it, and E the final 2 ML.

**SM5** is SM4 without carry-over. Here every event is large enough to absorb the negatives after it, so nothing needs
carrying and SM5 gives the same result as SM4.

**SM6** applies SM1 separately within each segment, here 1-20 Jan and 21 Jan onwards. The first segment has a higher
ratio of negative to positive flow (36 ML against 93 ML, versus 30 ML against 193 ML), so event A is reduced
proportionally more than the later events.

**SM7** gives each block of negatives to whichever of the events either side of it is larger. The 22 ML between
A (93 ML) and B (105 ML) goes to B, the 25 ML between C (64 ML) and D (15 ML) goes to C, and the 3 ML between D and
E (9 ML) goes to D. The negatives at the start and end go to the only neighbouring event (A and E), and no volume is
carried over.

**SM2, flow limit 5**: events are now runs of flow **above** 5 ML/d, and only the excess above 5 is reduced, so the
events differ from the Event column: e.g. the 5 on 5 Jan is no longer part of event A, and only the 7 on 3 Feb
remains of event D. The 4, 3 and 1 around it are kept, and no flow is reduced below 5 ML/d. The cost is that less
volume is available to absorb the negatives, so more are left unallocated (27 ML rather than 6 ML).

**SM2, gaps as zero**: NEGFLO6 read a missing measured flow as zero, which turns every missing day into a negative
residual equal to the modelled flow: here 188 ML of spurious negatives on top of the real 66 ML. The column shows 0.0
on those days (the clipped spurious negatives). The extra negatives wipe out event B and cut event C, and the residual
total itself drops from 220 ML to 32 ML. Excluding missing data avoids this.

## Using Negflo on your own data

Build a `Negflo` from your residual flows, or from measured and modelled flows, with one column per site and a date
index. Missing data should be NaN.

```python
negflo = Negflo(df_residual, flow_limit=0)                    # residual already calculated
negflo = Negflo.from_flows(df_measured, df_modelled)          # residual = measured - modelled
negflo = Negflo.from_flows(df_measured, df_modelled, segments=[("1970-07-01", "1990-06-30"),
                                                               ("1990-07-01", "2020-06-30")])

result = negflo.sm2()                    # a DataFrame, one column per site
negflo.unallocated_negatives             # {column: volume not redistributed by sm2}
negflo.to_file(out_filename="site")      # writes site.sm2
negflo.run_all("outputs")                # writes every method's result to the outputs folder
```

### Choosing a method

* **SM1** keeps the total volume and the variability of the flows, which suits a reach flowing into a storage.
  **SM6** does the same within each segment (e.g. water years), so negatives in one period don't reduce another.
* **SM2/SM3** suit negative flows that mainly occur on the rising limb of the hydrograph (they are put into the
  following event); **SM4/SM5** suit negatives on the falling limb (put into the preceding event). **SM7** puts each
  block of negatives into the larger neighbouring event.
* Carry-over (SM2, SM4) leaves less volume unallocated, but can move negatives a long way from where they occurred.
* A flow limit preserves small flows, at the cost of reducing larger flows more, and of more unallocated volume.
* Always check `unallocated_negatives`. If a lot of the residual is negative, first review the routing and delays, the
  gauge ratings, and whether the reach model needs a waterhole or losses.